# Session 1 — Pandas Foundations: Manufacturing Data Analysis

**Module:** M3 — Data Foundation  
**Duration:** ~2 hours live + 4–8 hours individual study  
**Prerequisites:** Python 1 & 2 essentials (variables, lists, dicts, loops, functions, files)  

---

## What we are building toward

By Session 15 we will ship a multi-agent AI system that investigates **real ZF Life manufacturing data**.
Before we can analyse data, we need to be able to *hold* it, *inspect* it, and *ask questions* about it.
Pandas is the tool that makes all of that possible.

This session is your practical Pandas launch. We start simple — a 4-row table, a tiny dataset —
and finish with a working Streamlit dashboard and a clear sense of how engineers actually use data.

---

## Session objectives

1. Understand what a Pandas DataFrame is and why it is useful in manufacturing
2. Load a manufacturing CSV file into Pandas
3. Inspect rows, columns, data types, and missing values
4. Select, filter, and sort production data
5. Create new calculated columns such as scrap rate
6. Group data by machine, shift, or defect type
7. Identify simple patterns in manufacturing data
8. Build a very small Streamlit app that displays a dataset and a simple summary
9. Continue studying independently using structured practice tasks

---

## Part 0 — Why Pandas for Manufacturing Data?

**Time:** 10 minutes

Manufacturing companies generate data from machines, inspections, operators, materials, shifts, and production lines.
Pandas helps analysts answer practical questions such as:

- Which machine has the highest scrap rate?
- Which shift produces the most defects?
- Are defective units increasing over time?
- Which batches should be investigated?
- Is the data clean enough to trust?

Without Pandas, answering these questions means writing complex loops, managing lists of lists, and manually computing averages.
With Pandas, each question above is one or two lines of code.

> **Mentor note:** Pandas is not just for manipulating tables. It is a tool for asking better questions about
> production, quality, machines, shifts, defects, and business decisions.

### What is a DataFrame?

A **DataFrame** is a table. Each row is a record. Each column is a variable.
You already know this from Excel. Pandas gives you that same structure inside Python — but with code instead of clicks.

| batch_id | date       | shift   | machine | units_produced | defective_units |
|----------|------------|---------|---------|---------------:|----------------:|
| B001     | 2026-01-05 | Morning | M1      |            500 |              12 |
| B002     | 2026-01-05 | Morning | M2      |            460 |              18 |
| B003     | 2026-01-05 | Evening | M1      |            520 |               9 |
| B004     | 2026-01-06 | Night   | M3      |            430 |              25 |

One row = one production batch record.  
Let us load our production CSV and start asking questions.

In [ ]:
import pandas as pd

df = pd.read_csv("data/production_log.csv")

df.head(4)

### Practical questions engineers can ask

Now that we have the table, let us answer three real engineering questions with code.

**Question A:** Which machine has the highest number of defective units?  
**Question B:** Which shift has the highest number of defective units?  
**Question C:** Which batch should be reviewed first?

In [ ]:
# Question A: Which machine has the highest number of defective units?
machine_defects = df.groupby("machine")["defective_units"].sum()
print(machine_defects)
print("Answer:", machine_defects.idxmax())

In [ ]:
# Question B: Which shift has the highest number of defective units?
shift_defects = df.groupby("shift")["defective_units"].sum()
print(shift_defects)
print("Answer:", shift_defects.idxmax())

In [ ]:
# Question C: Which batch should be reviewed first?
batch_to_review = df.sort_values("defective_units", ascending=False).iloc[0]
print(batch_to_review[["batch_id", "machine", "shift", "defective_units"]])

### Key lesson

Pandas is like a smart spreadsheet inside Python.
It helps engineers inspect production data, ask better questions,
and find where investigation should start.

> **Mini challenge:** Run the three question cells above. Then filter `df` to only Machine M3 rows and re-run Question A manually.
> Does the answer change when you restrict to a single machine?

---

## Part 1 — Inspecting the Data in Detail

**Time:** 20 minutes

We already loaded the CSV above. Now let us slow down and properly inspect what we have.
These inspection tools should be the **first thing you run** every time you open a new dataset.

### Concepts

- previewing rows with `df.head()`
- checking dimensions with `df.shape`
- listing column names with `df.columns`
- checking data types and nulls with `df.info()`
- counting missing values with `df.isna().sum()`

Pay attention to columns where the dtype is `object` but you expected a number or a date.
That is almost always a sign that something went wrong during data collection.

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Look at the actual rows where defective_units is missing
df[df["defective_units"].isna()]

### Exercise

Answer these questions using the output from the cells above:

1. How many production records are in the dataset?
2. How many columns does the dataset have?
3. Which columns are numeric?
4. Are there missing values? In which columns?
5. What does one row represent?

---

## Part 2 — Selecting, Filtering, and Sorting Production Data

**Time:** 25 minutes

Now that we can see the data, let us learn to extract exactly what we need.

### Concepts

- selecting a single column
- selecting multiple columns
- filtering rows with a condition
- combining multiple conditions with `&` and `|`
- sorting with `sort_values()`

### Basic syntax

```python
df["machine"]                                            # single column
df[["machine", "shift", "units_produced"]]               # multiple columns
df[df["machine"] == "M1"]                               # filter by value
df[df["defective_units"] > 15]                          # filter by condition
df[(df["machine"] == "M2") & (df["shift"] == "Morning")] # multiple conditions
df.sort_values("defective_units", ascending=False)       # sort
```

In [ ]:
# The "machine" column shows which machine produced each batch in the log
df["machine"]

In [ ]:
# This expression does not create a completely independent DataFrame in memory.
# In pandas, df[["machine", "shift", "units_produced"]] returns a *view-like* subset
# (a new DataFrame object that still references the same underlying data buffer
# whenever possible). This means:
# - It is a DataFrame with only the selected columns.
# - It usually does NOT copy all the data: changes to the original df's column
#   values may be reflected here, and vice versa, depending on the operation.
# - To force an entirely separate DataFrame with its own memory, use .copy():
#       df_subset = df[["machine", "shift", "units_produced"]].copy()
df[["machine", "shift", "units_produced"]]

In [ ]:
df[df["machine"] == "M1"]

In [ ]:
df[df["defective_units"] > 15]

In [ ]:
# Multiple conditions: use & (and) or | (or), and wrap each condition in parentheses
df[(df["machine"] == "M2") & (df["shift"] == "Morning")]

In [ ]:
df.sort_values("defective_units", ascending=False).head(5)

### Manufacturing questions

Use the techniques above to answer:

1. Which records belong to Machine M1?
2. Which batches had more than 15 defective units?
3. Which production record had the highest number of defects?
4. Which night-shift records should be reviewed?

> **Challenge:** Filter for batches where `defective_units` is greater than 20 **and** the shift is `"Night"`.
> How many such batches are there?

---

## Part 3 — Creating Manufacturing Metrics

**Time:** 20 minutes

Raw defect counts are useful, but they can be misleading.
A machine that produced 1 000 units with 50 defects is better than one that produced 200 units with 30 defects.

The metric that makes this comparison fair is the **scrap rate**:

```
scrap_rate = defective_units / units_produced
```

Pandas lets us create new columns by performing operations on existing ones — just like Excel formulas, but in code.

### Concepts

- creating new columns with arithmetic operations
- rounding with `.round()`
- using the new column to find the worst-performing batch
- understanding why percentage metrics are more useful than raw counts

In [ ]:
kpi_series = df["defective_units"] / df["units_produced"]

df["scrap_rate"] = kpi_series

df.head(5)

In [ ]:
df["scrap_rate_percent"] = df["scrap_rate"] * 100

df["scrap_rate_percent"] = df["scrap_rate_percent"].round(2)

df[["batch_id", "machine", "shift", "units_produced", "defective_units", "scrap_rate_percent"]].head(10)

In [ ]:
# Which batch has the highest scrap rate?
df.sort_values("scrap_rate_percent", ascending=False).head(5)

In [ ]:
# This line is doing three main things in a row (that's what "method chaining" means):
#
# 1. df[df["scrap_rate"] > 0.05]
#    - Looks at the `scrap_rate` column of the DataFrame `df`.
#    - Keeps only the rows where `scrap_rate` is greater than `0.05`.
#    - The result is a smaller DataFrame containing only "high scrap rate" rows.
#
# 2. .groupby("machine")
#    - Takes that filtered DataFrame and groups the rows by the value in the `machine` column.
#    - So all rows for the same machine (M1, M2, etc.) are collected into their own group.
#
# 3. ["scrap_rate"].describe()
#    - From each machine group, it selects only the `scrap_rate` column.
#    - Then `describe()` calculates summary statistics for `scrap_rate` within each machine group:
#      - count (how many rows)
#      - mean (average scrap rate)
#      - std (standard deviation)
#      - min (minimum scrap rate)
#      - 25% / 50% / 75% (quartiles, including median)
#      - max (maximum scrap rate)
#
# Overall, this code:
# - Filters to only rows where scrap rate is above 5%,
# - Groups those rows by machine,
# - And then shows a table of summary statistics of scrap rate for each machine.
df[df["scrap_rate"] > 0.05].groupby("machine")["scrap_rate"].describe()

### Exercise

1. Which batch has the highest scrap rate?
2. Is the batch with the most defects always the batch with the worst scrap rate?
3. Why is percentage often more useful than raw defect count?

> **Challenge:** Create a new column called `good_units` equal to `units_produced - defective_units`.
> Then create a column `is_high_scrap` that is `True` where `scrap_rate` is greater than `0.05`.

---

## Part 4 — Grouping by Machine and Shift

**Time:** 25 minutes

Individual rows tell you about single batches. Grouped summaries tell you about *patterns*.
This is where Pandas becomes truly powerful for manufacturing analysis.

`groupby()` splits the DataFrame into groups, and `agg()` computes statistics for each group.

### Concepts

- `groupby()` — split by category
- `agg()` — compute multiple aggregations at once: `sum`, `mean`, `count`
- sorting grouped results to find the worst/best performer

```python
df.groupby("machine").agg({
    "units_produced": "sum",
    "defective_units": "sum",
    "scrap_rate_percent": "mean"
})
```

In [ ]:
machine_summary = df.groupby("machine").agg({
    "units_produced": "sum",
    "defective_units": "sum",
    "scrap_rate_percent": "mean"
})

machine_summary

In [ ]:
shift_summary = df.groupby("shift").agg({
    "units_produced": "sum",
    "defective_units": "sum",
    "scrap_rate_percent": "mean"
}).sort_values("scrap_rate_percent", ascending=False)

shift_summary

### Manufacturing questions

1. Which machine produced the most units?
2. Which machine had the most defective units?
3. Which machine had the highest average scrap rate?
4. Which shift looks most suspicious?
5. What further data would we need before blaming a machine or shift?

> **Important:** A high scrap rate can suggest where to investigate, but it does not prove the root cause by itself.
> Students should learn not to overclaim.

> **Challenge:** Can you produce a summary grouped by **both** machine and shift at the same time?  
> Hint: `df.groupby(["machine", "shift"])`

---

## Part 5 — First Mini Streamlit App

**Time:** 15 minutes

So far all our work lives in this notebook. A Streamlit app turns your analysis into a simple
interactive web page that anyone on the team can use — no Python knowledge required on their end.

The code below is a complete Streamlit app. Save it as `app.py` and run it with:

```bash
streamlit run app.py
```

### Concepts

- `st.title()` — page title
- `st.subheader()` — section heading
- `st.dataframe()` — display a Pandas DataFrame as an interactive table
- `st.selectbox()` — dropdown filter
- Connecting a filter value to a filtered DataFrame

In [ ]:
# Save this as app.py and run:  streamlit run app.py

# import streamlit as st
# import pandas as pd
#
# st.title("Manufacturing Production Dashboard")
#
# df = pd.read_csv("data/production_log.csv")
#
# st.subheader("Raw Production Data")
# st.dataframe(df)
#
# df["scrap_rate_percent"] = (
#     df["defective_units"] / df["units_produced"] * 100
# ).round(2)
#
# st.subheader("Machine Summary")
# machine_summary = df.groupby("machine").agg({
#     "units_produced": "sum",
#     "defective_units": "sum",
#     "scrap_rate_percent": "mean"
# })
#
# st.dataframe(machine_summary)
#
# selected_machine = st.selectbox(
#     "Select a machine",
#     df["machine"].unique()
# )
#
# filtered_df = df[df["machine"] == selected_machine]
#
# st.subheader(f"Records for Machine {selected_machine}")
# st.dataframe(filtered_df)

print("Copy the commented code above into a file called app.py and run: streamlit run app.py")

### Learning outcome

Students understand that Pandas can power simple dashboards and interactive analysis tools.
Any notebook analysis you write here can become a shareable app with very little extra code.

---

## Part 6 — Mini-Project Brief

**Time:** 5 minutes introduction during live session  
**Recommended completion time:** 2–4 hours individually  

### Scenario

A factory manager is worried that scrap is increasing in the production process.
You receive a simplified production dataset and must use Pandas to investigate machine and shift performance.

### Part 1 — Jupyter Notebook Analysis

Create a notebook that:
1. Loads the manufacturing dataset
2. Shows the first rows
3. Checks data types and missing values
4. Calculates scrap rate percentage
5. Finds the batch with the highest scrap rate
6. Groups results by machine
7. Groups results by shift
8. Writes 3–5 observations in markdown

### Part 2 — Streamlit Dashboard

Create a simple Streamlit app that includes:
1. A title and short project description
2. The raw dataset
3. A machine-level summary table
4. A shift-level summary table
5. A machine filter
6. A short conclusion section

### Part 3 — Business Interpretation

Answer:
1. Which machine should be investigated first?
2. Which shift has the highest average scrap rate?
3. What additional data would help confirm the root cause?
4. Why should we avoid blaming an operator or machine based only on this small dataset?

---

### Suggested Mini-Project Dataset Columns

| Column          | Description                |
|-----------------|----------------------------|
| batch_id        | Unique batch code          |
| date            | Production date            |
| shift           | Morning, Evening, or Night |
| machine         | Machine ID                 |
| operator        | Operator code              |
| units_produced  | Number of produced units   |
| defective_units | Number of defective units  |
| defect_type     | Main defect category       |
| material_batch  | Material batch code        |

Optional advanced columns:

| Column      | Description                 |
|-------------|-----------------------------|
| temperature | Machine/process temperature |
| pressure    | Process pressure            |
| cycle_time  | Time per production cycle   |
| humidity    | Environmental humidity      |

---

## Exercises for Individual Study

### Practice Set 1 — Data Inspection

1. Load a new CSV file
2. Display the first 10 rows
3. Show the number of rows and columns
4. List all column names
5. Check data types
6. Find missing values

### Practice Set 2 — Filtering

1. Show only records from Machine M1
2. Show only night-shift records
3. Show records where defective units are greater than 20
4. Show records where scrap rate is above 5%
5. Show records for Machine M2 during the evening shift

### Practice Set 3 — Metrics

Create:
1. `scrap_rate`
2. `scrap_rate_percent`
3. `good_units` — units produced minus defective units
4. `is_high_scrap` — True where scrap rate is greater than 5%

### Practice Set 4 — Grouping

Calculate:
1. Total units by machine
2. Total defective units by machine
3. Average scrap rate by machine
4. Average scrap rate by shift
5. Most common defect type by machine (if the column exists)

### Practice Set 5 — Streamlit Extension

Improve your app by adding:
1. A shift filter
2. A defect type filter
3. A chart showing defective units by machine
4. A chart showing average scrap rate by shift
5. A warning message when scrap rate is above a chosen threshold

---

## Recommended Self-Study Path After Session 1

### Step 1 — Pandas Core Skills

Study:
- selecting rows and columns
- filtering
- sorting
- missing values
- creating columns
- grouping and aggregation
- merging datasets
- working with dates

### Step 2 — Manufacturing Analysis Skills

Practice:
- scrap rate analysis
- defect type analysis
- machine comparison
- shift comparison
- production trend analysis
- suspicious batch detection

### Step 3 — Jupyter Notebook Reporting

Learn to:
- write markdown explanations
- organise notebooks clearly
- separate code, output, and interpretation
- present business conclusions

### Step 4 — Streamlit Apps

Learn to:
- display dataframes
- add filters
- add metrics
- add charts
- structure a simple dashboard

---

## Suggested Course Flow Beyond Session 1

| Module | Topic |
|--------|-------|
| Module 1 | Pandas Foundations — DataFrames, Series, loading CSV, inspecting |
| Module 2 | Cleaning Manufacturing Data — missing values, duplicates, invalid values |
| Module 3 | Filtering and Investigating — boolean filters, sorting, suspicious batches |
| Module 4 | Grouping and Aggregation — machine summaries, shift summaries |
| Module 5 | Joining Tables — production, inspection, material, machine tables |
| Module 6 | Time-Based Analysis — dates, trends, rolling averages, drift |
| Module 7 | Visualization — bar charts, line charts, defect distribution |
| Module 8 | Streamlit Dashboard — filters, summary cards, charts, conclusions |
| Module 9 | Mini Root-Cause Analysis Project — full investigation workflow |

---

## Final Student Deliverables

Each student should submit:
1. A Jupyter Notebook with the analysis
2. A Streamlit app file
3. The dataset used
4. A short written conclusion with 3–5 findings
5. One recommendation for further investigation

---

## Simple Evaluation Rubric

| Criteria | Points |
|---|---:|
| Loads and inspects dataset correctly | 20 |
| Calculates scrap rate correctly | 20 |
| Uses filtering and grouping correctly | 20 |
| Creates a working Streamlit app | 20 |
| Provides clear business observations | 20 |

**Total: 100 points**